# 05. H1b — позиция целевого, независимая от повтора

**Зачем.** В H1 позиция целевого (1, 5, 9) была жёстко связана с номером повтора, а у каждого повтора свои соседи. Поэтому различия по позиции нельзя было отделить от различий в наборах соседей.

**Дизайн.** Для каждого целевого берём один набор из 8 соседей в фиксированном порядке и вставляем целевой на позиции 1, 5 и 9. Внутри такой тройки батчи различаются **только позицией целевого**: те же тексты в том же порядке. Доля токсичных соседей: 0 / 50 / 100 %.

300 целевых × 3 уровня × 3 позиции × `N_SETS` наборов (по умолчанию 1) = **2 700 батчей**. Одиночные запросы не нужны: берём их из прогона H1 (`h1_singles_<модель>_<язык>.parquet`).

**Порядок:** сначала `PILOT = True` (20 целевых, 180 батчей), затем `PILOT = False`. Ответы кэшируются, прерванный запуск можно продолжить.

In [1]:
MODEL_KEY = "gigachat_lite"   # та же модель, что в основном прогоне H1
LANG = "en"
PILOT = False                  # True — 20 целевых; False — все 300
N_PILOT = 20
N_SETS = 1                    # сколько независимых наборов соседей на цель (1 хватает: позиции внутри набора сравниваются напрямую)
LEVELS = (0, 4, 8)            # токсичных из 8 соседей: 0 / 50 / 100 %
POSITIONS = (1, 5, 9)

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
DATA_PROC = PROJECT_DIR / "data" / "processed"
DATA_RES = PROJECT_DIR / "data" / "results"
DATA_RES.mkdir(parents=True, exist_ok=True)

from src.batching import build_position_plan
from src.config import MODELS
from src.llm import ask_many
from src.prompts import SYSTEM, build_user_prompt, parse_labels, p_toxic_from_logprobs

targets = pd.read_parquet(DATA_PROC / "targets.parquet")
neighbors = pd.read_parquet(DATA_PROC / "neighbors_pool.parquet")
text = pd.concat([targets, neighbors]).set_index("id")["text"]
system = SYSTEM[LANG]

plan = build_position_plan(targets, neighbors, levels=LEVELS, positions=POSITIONS, n_sets=N_SETS)
if PILOT:
    pilot_ids = (targets.groupby("target_group", group_keys=False)
                        .apply(lambda g: g.sample(N_PILOT // 2, random_state=0))["id"])
    plan = plan[plan["target_id"].isin(pilot_ids)]
suffix = f"{MODEL_KEY}_{LANG}" + ("_pilot" if PILOT else "")
plan.to_parquet(DATA_PROC / f"h1b_plan_{suffix}.parquet", index=False)
print(MODELS[MODEL_KEY])
print(f"батчей: {len(plan)}; целевых: {plan['target_id'].nunique()}")
plan.head()

{'provider': 'gigachat', 'model': 'GigaChat-2', 'logprobs': False, 'workers': 1}
батчей: 2700; целевых: 300


,batch_id,target_id,n_toxic,prop_toxic,rep,target_pos,item_ids,length_tol
0,h1b_5903407_k0_s0_p1,5903407,0,0.0,0,1,"[5903407, 1049200, 5701090, 5339215, 6275122, ...",0.15
1,h1b_5903407_k0_s0_p5,5903407,0,0.0,0,5,"[1049200, 5701090, 5339215, 6275122, 5903407, ...",0.15
2,h1b_5903407_k0_s0_p9,5903407,0,0.0,0,9,"[1049200, 5701090, 5339215, 6275122, 6083226, ...",0.15
3,h1b_5903407_k4_s0_p1,5903407,4,0.5,0,1,"[5903407, 6268466, 555788, 5618198, 6083226, 5...",0.15
4,h1b_5903407_k4_s0_p5,5903407,4,0.5,0,5,"[6268466, 555788, 5618198, 6083226, 5903407, 5...",0.15


### Проверка плана
Внутри каждой тройки (цель, набор, уровень) три батча должны состоять из **одних и тех же** текстов и отличаться лишь позицией целевого. Длина батча не должна зависеть от позиции.

In [3]:
# одинаковый состав соседей и порядок при разных позициях
def neighbors_only(r):
    return tuple(i for i in r["item_ids"] if i != r["target_id"])
plan["neigh"] = plan.apply(neighbors_only, axis=1)
same = plan.groupby(["target_id", "rep", "n_toxic"])["neigh"].nunique().eq(1).all()
print("Состав и порядок соседей одинаковы для всех позиций:", same)
assert same

ntok = pd.concat([targets, neighbors]).set_index("id")["n_tokens"]
plan["batch_tokens"] = plan["item_ids"].map(lambda ids: int(ntok.loc[ids].sum()))
print(plan.groupby("target_pos")["batch_tokens"].mean().round(1).to_dict(), "— средняя длина по позициям")
print(plan.groupby(["n_toxic", "target_pos"]).size().unstack())

Состав и порядок соседей одинаковы для всех позиций: True
{1: 353.7, 5: 353.7, 9: 353.7} — средняя длина по позициям
target_pos    1    5    9
n_toxic                  
0           300  300  300
4           300  300  300
8           300  300  300


## Прогон батчей

In [4]:
def to_rows(resp, n, extra):
    labels, ok = parse_labels(resp["text"], n)
    probs = p_toxic_from_logprobs(resp["tokens"], n) if resp.get("tokens") else [None] * n
    u = resp.get("usage") or {}
    return dict(extra, labels=labels, probs=probs, parse_ok=ok, finish_reason=resp.get("finish_reason"),
                error=resp.get("error"), in_tokens=u.get("input"), out_tokens=u.get("output"), raw=resp["text"])

jobs = [dict(id=r.batch_id, system=system, user=build_user_prompt([text[i] for i in r.item_ids]), max_tokens=120)
        for r in plan.itertuples()]
resp = ask_many(MODEL_KEY, jobs, desc="батчи H1b")

rows = [to_rows(resp[r.batch_id], len(r.item_ids), dict(batch_id=r.batch_id)) for r in plan.itertuples()]
batches = plan.drop(columns=["neigh", "batch_tokens"]).merge(pd.DataFrame(rows), on="batch_id")
batches["y_batch"] = [lab[p - 1] for lab, p in zip(batches["labels"], batches["target_pos"])]
batches["p_batch"] = [pr[p - 1] for pr, p in zip(batches["probs"], batches["target_pos"])]
batches[["batch_id", "prop_toxic", "target_pos", "y_batch", "parse_ok", "finish_reason"]].head()

батчи H1b:   0%|          | 0/2700 [00:00<?, ?it/s]

,batch_id,prop_toxic,target_pos,y_batch,parse_ok,finish_reason
0,h1b_5903407_k0_s0_p1,0.0,1,1,True,stop
1,h1b_5903407_k0_s0_p5,0.0,5,1,True,stop
2,h1b_5903407_k0_s0_p9,0.0,9,1,True,stop
3,h1b_5903407_k4_s0_p1,0.5,1,1,True,stop
4,h1b_5903407_k4_s0_p5,0.5,5,1,True,stop


In [5]:
print("ошибки API:                 ", batches["error"].notna().sum())
print("ответ не распознан целиком: ", (~batches["parse_ok"]).mean().round(3))
print("нет метки целевого:         ", batches["y_batch"].isna().mean().round(3))
print("токенов на батч (вход / выход):", batches["in_tokens"].mean().round(0), "/", batches["out_tokens"].mean().round(1))
print("\nДоля нераспознанных по позиции (не должна различаться):")
print(batches.groupby("target_pos")["y_batch"].apply(lambda s: s.isna().mean()).round(3))

ошибки API:                  0
ответ не распознан целиком:  0.001
нет метки целевого:          0.0
токенов на батч (вход / выход): 411.0 / 59.5

Доля нераспознанных по позиции (не должна различаться):
target_pos
1    0.0
5    0.0
9    0.0
Name: y_batch, dtype: float64


In [6]:
batches.drop(columns=["raw"]).to_parquet(DATA_RES / f"h1b_batches_{suffix}.parquet", index=False)
print("Сохранено:", f"h1b_batches_{suffix}.parquet")
batches.groupby(["prop_toxic", "target_pos"])["y_batch"].mean().unstack().round(3)

Сохранено: h1b_batches_gigachat_lite_en.parquet


target_pos,1,5,9
prop_toxic,,,
0.0,0.660,0.617,0.593
0.5,0.663,0.487,0.520
1.0,0.653,0.427,0.523
